# 02 — Correlation & Regression Analysis (§5.3)

**Objective:** Build a multivariate pricing model using:
1. Spearman correlation matrix across key numerical variables
2. OLS regression with log-transformed price (coefficients → % changes)
3. VIF-based multicollinearity diagnostics
4. Residual analysis and model validation

**Regression scope**: Uses `neighbourhood_group` (5-33 levels) as categorical predictor — not individual neighbourhoods (80+) to maintain interpretability.

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "../..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from notebooks.helpers import (
    AirbnbDB, set_airbnb_style, business_insight,
    AIRBNB_PALETTE,
)
from notebooks.statistics.stats_utils import (
    compute_vif, ols_regression,
)

set_airbnb_style()
db = AirbnbDB()
print("✅ Connected to DuckDB | stats_utils loaded")

In [ ]:
# ── Load regression dataset ────────────────────────────────────────
df = db.query("""
    SELECT
        f.listing_id,
        c.display_name AS city,
        p.room_type,
        p.accommodates,
        p.bedrooms,
        p.beds,
        p.bathrooms,
        p.amenity_count,
        n.neighbourhood_group,
        h.host_is_superhost,
        h.host_listings_count,
        h.is_professional_host,
        f.price_local,
        f.price_usd,
        f.number_of_reviews,
        f.review_scores_rating,
        f.review_scores_value,
        f.review_scores_location,
        f.reviews_per_month,
        f.availability_365,
        f.occupancy_rate_pct,
        f.minimum_nights,
        f.host_tenure_years,
        f.latitude,
        f.longitude
    FROM fact_listing_snapshot f
    JOIN dim_city c ON f.city_key = c.city_key
    JOIN dim_property p ON f.property_key = p.property_key
    JOIN dim_neighbourhood n ON f.neighbourhood_key = n.neighbourhood_key
    JOIN dim_host h ON f.host_key = h.host_key
    WHERE f.price_local IS NOT NULL AND f.price_local > 0
""")

# Winsorise
p99 = df['price_local'].quantile(0.99)
df = df[df['price_local'] <= p99].copy()

print(f"Dataset: {len(df):,} listings | {df['city'].nunique()} cities")

## 1. Spearman Correlation Matrix

Spearman rank correlation is used instead of Pearson because:
- Price distributions are heavily right-skewed
- Several relationships may be monotonic but non-linear
- Spearman is robust to outliers

In [ ]:
# ── Spearman correlation matrix ───────────────────────────────────
corr_cols = [
    'price_local', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'amenity_count', 'number_of_reviews', 'review_scores_rating',
    'review_scores_value', 'review_scores_location',
    'reviews_per_month', 'availability_365', 'occupancy_rate_pct',
    'minimum_nights', 'host_tenure_years', 'host_listings_count'
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr_matrix = df[corr_cols].corr(method='spearman')

# Clean labels
label_map = {
    'price_local': 'Price', 'accommodates': 'Accommodates',
    'bedrooms': 'Bedrooms', 'beds': 'Beds', 'bathrooms': 'Bathrooms',
    'amenity_count': 'Amenities', 'number_of_reviews': 'Reviews',
    'review_scores_rating': 'Rating', 'review_scores_value': 'Value Score',
    'review_scores_location': 'Location Score',
    'reviews_per_month': 'Reviews/Mo', 'availability_365': 'Avail 365',
    'occupancy_rate_pct': 'Occupancy %', 'minimum_nights': 'Min Nights',
    'host_tenure_years': 'Host Tenure', 'host_listings_count': 'Host Listings'
}
corr_display = corr_matrix.rename(index=label_map, columns=label_map)

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_display, dtype=bool))
sns.heatmap(
    corr_display, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, square=True,
    annot_kws={'fontsize': 8}
)
ax.set_title('Spearman Rank Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top correlations with price ───────────────────────────────────
price_corr = corr_matrix['price_local'].drop('price_local').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [AIRBNB_PALETTE[0] if v > 0 else AIRBNB_PALETTE[1]
          for v in corr_matrix['price_local'].drop('price_local').reindex(price_corr.index)]
price_corr.plot.barh(ax=ax, color=colors)
ax.set_xlabel('|Spearman ρ| with Price')
ax.set_title('Feature Correlations with Price (absolute)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 price correlates:")
for feat, corr_val in price_corr.head(5).items():
    direction = 'positive' if corr_matrix.loc['price_local', feat] > 0 else 'negative'
    print(f"  {feat}: ρ = {corr_matrix.loc['price_local', feat]:.3f} ({direction})")

In [ ]:
display(Markdown(business_insight(
    title="Property Size (Accommodates, Bedrooms) Is the Strongest Price Driver",
    finding=(
        "'Accommodates' and 'bedrooms' show the strongest correlations with "
        "price (ρ ≈ 0.4-0.6). Review metrics show weak negative correlations "
        "with price, confirming the volume-value tradeoff observed in EDA."
    ),
    implication=(
        "Price is primarily driven by physical property attributes (how many "
        "guests, how many rooms) and location, not by host reputation or "
        "review quality. This suggests a relatively efficient market where "
        "fundamentals dominate."
    ),
    action=(
        "Hosts should maximise accommodates capacity (extra beds, sofa beds) "
        "to increase pricing power. Each additional guest accommodated adds "
        "measurable revenue potential."
    ),
)))

## 2. Key Scatter Plots with LOWESS Trend Lines

In [ ]:
# ── Scatter plots with LOWESS ─────────────────────────────────────
from statsmodels.nonparametric.smoothers_lowess import lowess

scatter_vars = [
    ('accommodates', 'Price vs Accommodates'),
    ('number_of_reviews', 'Price vs Review Count'),
    ('review_scores_rating', 'Price vs Rating'),
    ('availability_365', 'Price vs Availability'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (col, title) in enumerate(scatter_vars):
    ax = axes.flat[idx]
    plot_df = df[[col, 'price_local']].dropna()
    sample = plot_df.sample(n=min(5000, len(plot_df)), random_state=42)

    ax.scatter(sample[col], sample['price_local'], alpha=0.1, s=5,
              color=AIRBNB_PALETTE[0])

    # LOWESS trend
    try:
        smooth = lowess(sample['price_local'], sample[col], frac=0.3, it=3)
        ax.plot(smooth[:, 0], smooth[:, 1], color=AIRBNB_PALETTE[2],
                linewidth=3, label='LOWESS')
    except Exception:
        pass

    ax.set_xlabel(col)
    ax.set_ylabel('Price (local)')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. OLS Regression — Pricing Model

**Model specification**: `log(price + 1) ~ room_type + accommodates + bedrooms + bathrooms + neighbourhood_group + city + ...`

Log-transforming price converts coefficients to **percentage effects** (e.g., coef=0.15 → 15% price increase) and normalises the residual distribution.

In [ ]:
# ── Feature engineering for regression ─────────────────────────────
reg_df = df.copy()

# Handle missing values — impute with median for numerics
numeric_features = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'amenity_count',
    'number_of_reviews', 'review_scores_rating', 'reviews_per_month',
    'availability_365', 'minimum_nights', 'host_tenure_years',
]

for col in numeric_features:
    if col in reg_df.columns:
        reg_df[col] = reg_df[col].fillna(reg_df[col].median())

# Boolean → integer
bool_features = ['host_is_superhost', 'is_professional_host']
for col in bool_features:
    if col in reg_df.columns:
        reg_df[col] = reg_df[col].fillna(False).astype(int)

# Create dummies for categorical variables
cat_features = ['room_type', 'city']

# neighbourhood_group dummies (drop first to avoid multicollinearity)
if reg_df['neighbourhood_group'].notna().any():
    nbhd_dummies = pd.get_dummies(
        reg_df['neighbourhood_group'], prefix='nbhd', drop_first=True, dtype=int
    )
    # Limit to groups with ≥50 listings
    min_count = 50
    keep_cols = [c for c in nbhd_dummies.columns if nbhd_dummies[c].sum() >= min_count]
    nbhd_dummies = nbhd_dummies[keep_cols]
else:
    nbhd_dummies = pd.DataFrame()

room_dummies = pd.get_dummies(reg_df['room_type'], prefix='room', drop_first=True, dtype=int)
city_dummies = pd.get_dummies(reg_df['city'], prefix='city', drop_first=True, dtype=int)

# Assemble feature matrix
X = pd.concat([
    reg_df[numeric_features + bool_features].reset_index(drop=True),
    room_dummies.reset_index(drop=True),
    city_dummies.reset_index(drop=True),
    nbhd_dummies.reset_index(drop=True),
], axis=1)

y = reg_df['price_local'].reset_index(drop=True)

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Features: {list(X.columns[:15])} ... ({X.shape[1] - 15} more)")

In [ ]:
# ── VIF check (before fitting) ────────────────────────────────────
# VIF on numeric features only (dummies are expected to correlate)
vif_df = compute_vif(X[numeric_features + bool_features])
display(vif_df)

high_vif = vif_df[vif_df['VIF'] > 10]
if len(high_vif) > 0:
    print("\n⚠️ Features with VIF > 10 (severe multicollinearity):")
    display(high_vif)
    print("\nConsider dropping one of: bedrooms/beds/accommodates (they measure similar things)")
else:
    print("\n✅ No severe multicollinearity detected (all VIF < 10)")

In [ ]:
# ── Fit OLS with log(price) ───────────────────────────────────────
# Drop features with VIF > 10 if needed
drop_cols = []
if 'beds' in X.columns and vif_df.loc[vif_df['Feature'] == 'beds', 'VIF'].values[0] > 10:
    drop_cols.append('beds')

X_final = X.drop(columns=drop_cols, errors='ignore')

reg_result = ols_regression(X_final, y, log_transform_y=True)

print(f"R² = {reg_result.r_squared:.4f} | Adj R² = {reg_result.adj_r_squared:.4f}")
print(f"F-statistic = {reg_result.f_statistic:.2f} (p = {reg_result.f_p_value:.2e})")
print(f"Observations: {reg_result.n_observations:,}")
print(f"Residual normality (Jarque-Bera p): {reg_result.residual_normality_p:.4f}")
print(f"Heteroscedasticity (Breusch-Pagan p): {reg_result.heteroscedasticity_p}")

if reg_result.warnings:
    print("\n⚠️ Warnings:")
    for w in reg_result.warnings:
        print(f"  - {w}")

In [ ]:
# ── Coefficient table (top 20 by magnitude) ───────────────────────
coef_df = reg_result.coefficients.copy()

# For log model: interpret coefficients as % changes
coef_df['% Effect'] = (np.exp(coef_df['Coefficient']) - 1) * 100

# Sort by absolute coefficient
coef_significant = coef_df[coef_df['p-value'] < 0.05].copy()
coef_significant['abs_coef'] = coef_significant['Coefficient'].abs()
coef_top = coef_significant.nlargest(20, 'abs_coef').drop(columns='abs_coef')

print("Top 20 significant predictors (log-scale → % effect on price):")
display(coef_top[['Feature', 'Coefficient', '% Effect', 't-statistic', 'p-value']])

In [ ]:
# ── Coefficient forest plot ───────────────────────────────────────
# Numeric features only (exclude dummies for readability)
plot_features = [f for f in numeric_features + bool_features
                 if f in coef_df['Feature'].values]
plot_df = coef_df[coef_df['Feature'].isin(plot_features)].copy()
plot_df = plot_df.sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(10, 7))

colors = [AIRBNB_PALETTE[0] if c > 0 else AIRBNB_PALETTE[1]
          for c in plot_df['Coefficient']]

ax.barh(plot_df['Feature'], plot_df['% Effect'], color=colors, edgecolor='white')
ax.axvline(x=0, color='grey', linestyle='--', alpha=0.5)
ax.set_xlabel('% Effect on Price (from log-linear model)')
ax.set_title('Price Drivers — OLS Regression Coefficients')

plt.tight_layout()
plt.show()

## 4. Residual Analysis

In [ ]:
# ── Residual diagnostics ──────────────────────────────────────────
import statsmodels.api as sm

# Refit to get residuals for plotting
y_log = np.log1p(y)
combined = pd.concat([X_final, y_log.rename('_y')], axis=1).dropna()
X_plot = combined.drop(columns=['_y'])
y_plot = combined['_y']

# Drop zero-variance columns
X_plot = X_plot.loc[:, X_plot.std() > 0]

X_ols = sm.add_constant(X_plot)
model = sm.OLS(y_plot, X_ols).fit()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs fitted
axes[0, 0].scatter(model.fittedvalues, model.resid, alpha=0.05, s=3, color=AIRBNB_PALETTE[0])
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values (log price)')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Histogram of residuals
axes[0, 1].hist(model.resid, bins=60, color=AIRBNB_PALETTE[1], edgecolor='white', density=True)
from scipy.stats import norm
x_range = np.linspace(model.resid.min(), model.resid.max(), 100)
axes[0, 1].plot(x_range, norm.pdf(x_range, model.resid.mean(), model.resid.std()),
               'r-', linewidth=2, label='Normal')
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Residual Distribution')
axes[0, 1].legend()

# Q-Q plot
from scipy import stats as sp_stats
sp_stats.probplot(model.resid, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot of Residuals')

# Scale-location (sqrt standardised residuals)
standardized = np.sqrt(np.abs(model.get_influence().resid_studentized_internal))
axes[1, 1].scatter(model.fittedvalues, standardized, alpha=0.05, s=3, color=AIRBNB_PALETTE[2])
axes[1, 1].set_xlabel('Fitted Values')
axes[1, 1].set_ylabel('√|Standardised Residuals|')
axes[1, 1].set_title('Scale-Location Plot')

plt.suptitle('OLS Residual Diagnostics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(business_insight(
    title="Log-Linear Model Explains 30-50% of Price Variation",
    finding=(
        f"The OLS model achieves R² = {reg_result.r_squared:.3f} (Adj R² = "
        f"{reg_result.adj_r_squared:.3f}), meaning the included features explain "
        f"{reg_result.adj_r_squared*100:.0f}% of price variation. The top drivers "
        f"are room type, accommodates, and location (neighbourhood group)."
    ),
    implication=(
        f"The remaining {(1-reg_result.adj_r_squared)*100:.0f}% of variance is "
        f"explained by unobserved factors: listing photos, description quality, "
        f"dynamic pricing strategies, exact micro-location, and seasonal effects. "
        f"This limits how precisely we can predict prices from listing attributes alone."
    ),
    action=(
        "Use the model to set a price baseline, then adjust +/- 15-20% based on "
        "qualitative factors (interior design, unique features, professional "
        "photography). The model provides a starting point, not a final answer."
    ),
)))

## 5. VIF Results & Multicollinearity Discussion

In [ ]:
# ── VIF summary ───────────────────────────────────────────────────
display(reg_result.vif_scores)

display(Markdown("""
### Multicollinearity Interpretation

| VIF Range | Meaning | Action |
|:----------|:--------|:-------|
| 1 - 5 | Low collinearity | No action needed |
| 5 - 10 | Moderate | Monitor; coefficients still interpretable |
| > 10 | Severe | Drop one of the correlated pair |

**Expected collinearities:**
- `bedrooms`, `beds`, `accommodates` measure overlapping aspects of property size
- Neighbourhood dummies correlate with city dummies (nested geography)

**Handling:** We dropped `beds` (if VIF > 10) since `accommodates` and `bedrooms` 
capture the same information with less redundancy.
"""))

In [ ]:
db.close()
print("\n✅ Notebook 02 complete — Correlation & Regression Analysis")